# 01 · Retrieval and evaluation foundations

**Deck section 1** · slides 4–14

Retrieval determines which evidence reaches the model. Evaluation establishes whether that
process stays dependable as the system changes. This notebook makes both concrete: you will
run the four-stage pipeline with real numbers attached to each stage, watch the recall budget
get spent, and finish by executing the fault-isolation tree against a question the system
actually gets wrong.

**By the end you can**

- name the four stages and say what each one can and cannot fix
- measure the difference between Recall@N and Recall@k, and explain why only one of them is
  a ceiling
- attribute a bad answer to retrieval, ranking, packing or generation *with a trace* rather
  than a guess
- recognise the four failure signatures of section 1 by their symptom


In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "raglab" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from raglab.bootstrap import bootstrap
bootstrap(verbose=False)

import raglab
from raglab import viz, tables, catalog, metrics, retrieve, pipeline
viz.reset_figures("1."); tables.reset_tables("1.")

bundle, index, pipe = raglab.quickstart(**raglab.TUNED)

---

## 1.1 Retrieval shapes the response

### What the code is about to do

Four stages, three handoffs. Every failure you will debug today lives in one of the arrows,
not in the model. The code below runs one question through all four and prints what each
stage handed to the next.


In [ ]:
viz.flow(
    [("User query", "what was asked"),
     ("Retrieve candidate evidence", "cheap and wide · N in the hundreds"),
     ("Rank and select context", "expensive and narrow · k in single digits"),
     ("Generate grounded answer", "with citations and an abstention path")],
    title="Retrieval shapes the response",
    kicker="Section 1 · pipeline",
    caption="A retrieval system should surface the evidence needed to answer a request while "
            "minimising irrelevant or contradictory material.",
    source="Deck slide 5",
)

In [ ]:
# One two-hop question, instrumented at every handoff.
q = next(x for x in bundle.questions
         if x.hops == 2 and x.question_type == "inference" and x.persona == "analyst")
trace = pipe.run(q.query, qid=q.qid)

gold_map, unresolved = metrics.resolve_gold(q, pipe.chunks)
gold = {c for s in gold_map.values() for c in s}

print(f"QUESTION  {q.query}")
print(f"GOLD      {len(gold_map)} evidence items → {len(gold)} chunks that satisfy them\n")

stages = [
    ("01  user query", 1, "the request as typed"),
    ("02  candidates", len(trace.candidates),
     f"{len(gold & set(trace.candidate_ids))}/{len(gold_map)} gold items present"),
    ("03  packed context", len(trace.packed),
     f"{len(gold & set(trace.packed_ids))}/{len(gold_map)} gold items survived, "
     f"{sum(b['tokens'] for b in trace.packed)} tokens"),
    ("04  answer", 1, f"{len(trace.citations)} citations emitted"),
]
for name, n, note in stages:
    print(f"  {name:<20} n={n:<5} {note}")
print(f"\nANSWER    {trace.answer[:220]}")

Read the third line again. The candidate pool held the gold evidence; the packed context is
where some of it stopped. That distinction — *retrieved* versus *retrieved and kept* — is the
one this whole section exists to make visible, and it is the difference between fixing your
retriever and fixing your packing.

## 1.2 Retrieval defines the answer space

### What the code is about to do

The funnel only narrows. Nothing downstream can recover a document the first stage never
returned. Rather than assert that, the next cell measures the actual width of each stage on
our corpus and eval set.


In [ ]:
viz.funnel(
    [("Query", "what was asked", 1.0),
     ("Available corpus", "everything indexed", 0.86),
     ("Candidate pool", "what stage one returned", 0.64),
     ("Packed context", "what the model sees", 0.44),
     ("Answer", "the claim, with citations", 0.28)],
    title="Retrieval defines the answer space",
    kicker="Section 1 · mental model",
    caption="The model can only use evidence that is selected for its context window.",
    source="Deck slide 6",
)

In [ ]:
import pandas as pd

# Measure the funnel: how much of the corpus survives each stage, averaged over the eval set.
sample = [x for x in bundle.questions if x.question_type != "null"][:60]
widths = {"corpus": [], "candidates": [], "packed": []}
recalls = {"candidates": [], "packed": []}
total_chunks = len(pipe.chunks)

for x in sample:
    tr = pipe.run(x.query, qid=x.qid, acl_groups=bundle.personas.get(x.persona))
    gm, _ = metrics.resolve_gold(x, pipe.chunks)
    widths["corpus"].append(total_chunks)
    widths["candidates"].append(len(tr.candidates))
    widths["packed"].append(len(tr.packed))
    recalls["candidates"].append(metrics.evidence_recall_at_k(tr.candidate_ids, gm))
    recalls["packed"].append(metrics.evidence_recall_at_k(tr.packed_ids, gm))

rows = [
    ["Available corpus", f"{total_chunks:,}", "100%", "1.00", "everything the index holds"],
    ["Candidate pool (N)", f"{sum(widths['candidates'])/len(sample):.0f}",
     f"{sum(widths['candidates'])/len(sample)/total_chunks:.1%}",
     f"{sum(recalls['candidates'])/len(sample):.3f}",
     "the ceiling — nothing below can exceed it"],
    ["Packed context (k)", f"{sum(widths['packed'])/len(sample):.1f}",
     f"{sum(widths['packed'])/len(sample)/total_chunks:.2%}",
     f"{sum(recalls['packed'])/len(sample):.3f}",
     "what the model actually reads"],
]
tables.show(pd.DataFrame(rows, columns=["Stage", "Width (avg)", "Share of corpus",
                                        "Evidence recall", "What it means"]),
            title="The funnel, measured on 60 answerable questions",
            kicker="Measured",
            caption="Stage one keeps under a tenth of the corpus and finds most of the gold. "
                    "Packing keeps under one percent — and that is where recall is lost.",
            emphasize="Evidence recall")

## 1.3 The pipeline is a recall budget spent on precision

This is the mental model to carry all day, and the single most useful thing in section 1.
Stage one *buys* recall cheaply. Stage two *spends* it, converting recall into precision.
Stage three packs a scarce token budget. Stage four turns evidence into a claim.


In [ ]:
viz.stages([
    dict(stage="Stage 1 · cheap", name="First-stage retrieval", tone="cool",
         body="Buys recall. Optimise Recall@N with N in the hundreds. Anything lost here is "
              "lost permanently.",
         knob="N, hybrid weights, chunking"),
    dict(stage="Stage 2 · expensive", name="Reranking", tone="amber",
         body="Converts recall into precision. Cannot exceed the ceiling set by stage one.",
         knob="model class, candidate depth"),
    dict(stage="Stage 3 · scarce", name="Context packing", tone="hot",
         body="A fixed token budget. Every distractor admitted costs a slot a gold chunk "
              "could have used.",
         knob="k, dedup, ordering"),
    dict(stage="Stage 4 · judged", name="Generation", tone="green",
         body="Turns evidence into a claim. Grounding failures here are independent of "
              "retrieval quality.",
         knob="instructions, abstention, citations"),
], title="The pipeline is a recall budget spent on precision",
   kicker="Mental model",
   caption="The most common architectural mistake: tuning stage two or three to fix a recall "
           "problem created in stage one. Measure Recall@N before you touch the reranker.",
   source="Deck slide 8")

### The ceiling, measured rather than asserted

"Reranking cannot exceed the ceiling set by stage one" is easy to nod at and easy to forget
at 2am. So let us put a number on it: sweep N, and watch the ceiling move while k stays
fixed.


In [ ]:
ns = [10, 25, 50, 100, 200, 400]
ceiling, delivered = [], []

for n in ns:
    v = pipe.variant(f"N={n}", n_candidates=n)
    rows = pipeline.evaluate(v, sample, pipe.chunks, personas=bundle.personas)
    ceiling.append(sum(r["evidence_recall_at_N"] for r in rows) / len(rows))
    delivered.append(sum(r["evidence_recall"] for r in rows) / len(rows))

viz.lines(ns, {"Evidence Recall@N  (the ceiling stage 1 buys)": ceiling,
               "Evidence Recall@k  (what actually reaches the model)": delivered},
          title="Stage one sets a ceiling; stages two and three spend what is under it",
          kicker="Measured · 60 questions",
          xlabel="N — first-stage candidate depth", ylabel="evidence recall",
          caption="The gap between the two lines is precision work: reranking and packing. "
                  "The upper line is what no amount of that work can exceed.")

In [ ]:
gap = [c - d for c, d in zip(ceiling, delivered)]
tables.show(pd.DataFrame({
    "N": ns,
    "Recall@N (ceiling)": [round(c, 3) for c in ceiling],
    "Recall@k (delivered)": [round(d, 3) for d in delivered],
    "Unspent recall": [round(g, 3) for g in gap],
    "Reading": ["stage 1 is the bottleneck — widen N first" if g < 0.10 else
                "stage 1 is fine — the loss is in ranking and packing" for g in gap],
}), title="Where is the loss, at each candidate depth?",
   kicker="Diagnosis",
   caption="Small N: the ceiling itself is low, so widen the first stage. Large N: the "
           "ceiling is high and the gap is wide — the money is now in stage two and three.",
   emphasize="Reading")

That table is a diagnostic procedure, not a result. On a client engagement you build it in
the first week and it tells you which half of the pipeline to spend the next three weeks on.
A team that skips it spends those weeks swapping embedding models to fix a packing problem.

## 1.4 Three ways to retrieve

Three signals, one candidate set. Most production systems run at least two.


In [ ]:
viz.flow([("Lexical", "terms, identifiers, error codes — exact and explainable"),
          ("Semantic", "meaning and paraphrase — bridges vocabulary the query does not share"),
          ("Grep", "files, logs and code — literal, inspectable, unranked")],
         title="Three ways to retrieve, one candidate set",
         kicker="Section 1 · methods", highlight=99,
         caption="Different search methods contribute complementary signals. The interesting "
                 "question is never which is best, it is which fails on your corpus.",
         source="Deck slide 7")

In [ ]:
from raglab.retrieve import GrepRetriever, RetrievalConfig

grep = GrepRetriever(pipe.chunks)
probe = "What is the recommended fix for ERR_CONN_RESET?"
cfg = RetrievalConfig(n_candidates=8, k=8)

legs = {
    "lexical (BM25)": index.lexical(probe, n=8),
    "semantic (dense)": retrieve.DenseRetriever(index, pipe.embedder).search(probe, 8, cfg),
    "grep (regex)": grep.search(r"ERR_CONN_RESET", n=8),
}
for name, hits in legs.items():
    print(f"\n{name.upper()}")
    for h in hits[:4]:
        print(f"  {h.score:8.4f}  {h.doc_id:<10} {h.title[:56]}")

ids = {name: {h.chunk_id for h in hits} for name, hits in legs.items()}
print("\nOVERLAP of the top 8")
names = list(ids)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        print(f"  {a:<18} ∩ {b:<18} = {len(ids[a] & ids[b])}")
print(f"  union of all three = {len(set().union(*ids.values()))} distinct chunks")

Three methods, and their top-8 lists barely intersect. That is the entire argument for hybrid
retrieval in one output: each leg is finding evidence the others miss, so a union is
strictly more informative than any single ranking. Notebook 04 turns that observation into a
fusion rule and measures what it is worth.

Note what grep does here. Perfect precision on the identifier, zero recall for the
paraphrase, and every result inspectable. For an agent working on a repository that trade is
usually the right one — which is why two shipped code assistants made opposite architectural
choices on exactly this question (notebook 04, §4.8).

## 1.5 Evaluation makes change visible

Three complementary lenses. None is sufficient alone, and the deck is specific about why.


In [ ]:
viz.flow([
    ("Offline evaluation", "fixed datasets · compare systems before release · deterministic, "
                           "seconds, no model calls"),
    ("Online evaluation", "production behaviour · failures, regressions, and the signals a "
                          "user actually feels"),
    ("LLM-as-a-judge", "one rubric applied at scale · qualitative properties string matching "
                       "cannot reach"),
], title="Three lenses, none sufficient alone", kicker="Section 1 · evaluation",
   highlight=99,
   caption="Offline tells you what changed before you ship. Online tells you what your users "
           "experienced. The judge scales the qualitative review neither of the others reaches.",
   source="Deck slide 9")

### Evaluate each handoff, not just the ends

Three handoffs, three separate scorecards. An end-to-end number tells you something moved. It
never tells you which stage moved it — and that is the difference between a debuggable system
and a mystery.


In [ ]:
rows = pipeline.evaluate(pipe, bundle.questions, pipe.chunks, personas=bundle.personas)
answerable = [r for r in rows if not r["is_null"]]

handoffs = pd.DataFrame([
    ["1 · query → retrieved results", "coverage, relevance, ranking",
     f"Evidence Recall@N {sum(r['evidence_recall_at_N'] for r in answerable)/len(answerable):.3f}"
     f" · nDCG@k {sum(r['ndcg'] for r in answerable if r['ndcg'] is not None)/len(answerable):.3f}"
     f" · MRR {sum(r['mrr'] for r in answerable if r['mrr'] is not None)/len(answerable):.3f}",
     "First-stage retrieval"],
    ["2 · retrieved results → LLM context", "selection, ordering, context fit",
     f"Evidence Recall@k {sum(r['evidence_recall'] for r in answerable)/len(answerable):.3f}"
     f" · full-chain {sum(r['full_chain_recall'] for r in answerable)/len(answerable):.3f}"
     f" · context precision "
     f"{sum(r['context_precision'] for r in answerable)/len(answerable):.3f}",
     "Reranking and packing"],
    ["3 · LLM answer → user query", "correctness, grounding, completeness",
     f"answer correctness {sum(r['answer_correct'] for r in answerable)/len(answerable):.3f}"
     f" · abstention recall "
     f"{metrics.abstention_scores(rows)['abstention_recall']:.3f}",
     "Generation controls"],
], columns=["Handoff", "What it measures", "Measured on this run", "Stage it indicts"])

tables.show(handoffs, title="Three handoffs, three scorecards",
            kicker="Layered evaluation",
            caption="Same questions, three independent readings. When one moves and the "
                    "others do not, you have located your regression.",
            source="Deck slide 10", emphasize="Measured on this run")

Look at the third row against the first two. Retrieval is finding roughly four fifths of the
gold evidence and answer correctness is far lower. Under an end-to-end score that gap is
invisible; under three scorecards it is a diagnosis, and section 1.6 is where we make it a
formal one.

*(For the record: the offline reader in this toolkit is extractive — it quotes evidence and
cannot derive a comparison or an ordering. Some of that gap is genuinely the reader, and
notebook 06 attributes it precisely rather than assuming.)*

---

## 1.6 The fault-isolation tree, executed

### What the code is about to do

The deck draws this tree as a poster. Here it is a function. Four questions, evaluated
against a real failing example, returning a single owning stage — and a trace of how it got
there, so the verdict is auditable rather than merely plausible.

Each question is a predicate over one dictionary:

```python
{"gold_in_packed":        did any gold evidence reach the model?
 "gold_in_candidates":    was every gold chunk in the top-N pool?
 "gold_intact_in_context": did the packed context keep it, intact and attributed?
 "answer_entailed":       is the answer actually supported by what was packed?}
```


In [ ]:
catalog.FAULT_ISOLATION.figure(
    caption="Four questions get you to a single owning stage. Walk it against one failing "
            "example before you touch any configuration.")

In [ ]:
catalog.FAULT_ISOLATION.show_table()

### Build the context the tree needs

Everything the tree asks for is already in the trace. That is not a coincidence — it is why
the deck insists the query path records retrieved items, scores, selected context and the
response. No trace, no tree.


In [ ]:
def isolate(question, pipeline_obj, chunks, judge=None):
    '''Run one question and answer the tree's four predicates from its trace.'''
    tr = pipeline_obj.run(question.query, qid=question.qid,
                          acl_groups=bundle.personas.get(question.persona))
    gm, _ = metrics.resolve_gold(question, chunks)
    gold = {c for s in gm.values() for c in s}
    packed, cands = set(tr.packed_ids), set(tr.candidate_ids)

    # "Entailed" is the one predicate that needs a reader. Two checks, not one:
    # faithfulness (every claim maps to a span in a cited block) AND completeness (the
    # answer resolves the question that was asked). The deck draws Q4 as entailment alone
    # because it assumes an answer that at least attempts the question. A reader that
    # quotes a true but irrelevant sentence passes faithfulness and fails the user, and
    # without the second check the tree sends you off to blame the label.
    from raglab.judge import HeuristicJudge
    judge = judge or HeuristicJudge()
    v = judge.judge_all(question.query, tr.answer, tr._packed_obj)
    entailed = (v["faithfulness"].passed and v["completeness"].passed
                and not metrics.abstained(tr.answer))

    ctx = {
        "gold_in_packed": bool(gold & packed),
        "gold_in_candidates": all(s & cands for s in gm.values()) if gm else True,
        "gold_intact_in_context": all(s & packed for s in gm.values()) if gm else True,
        "answer_entailed": entailed,
        "_trace": tr, "_gold": gm,
    }
    return ctx, tr


# Find a question the system gets wrong, and isolate it.
failing = next(r for r in rows
               if not r["is_null"] and r["full_chain_recall"] == 0.0 and r["gold_items"] >= 2)
question = next(x for x in bundle.questions if x.qid == failing["qid"])

ctx, tr = isolate(question, pipe, pipe.chunks)
print(f"FAILING QUESTION  {question.query}")
print(f"GOLD ANSWER       {question.answer}")
print(f"SYSTEM ANSWER     {tr.answer[:180]}\n")
for key in ("gold_in_packed", "gold_in_candidates", "gold_intact_in_context",
            "answer_entailed"):
    print(f"  {key:<26} {ctx[key]}")

In [ ]:
verdict = catalog.FAULT_ISOLATION.explain(ctx)

### The same tree over the whole eval set

One verdict is an anecdote. Run the tree over every failing question and the output is a
work plan: the distribution of owning stages tells you where the next three weeks go. This
is exactly what interview question Q1 is asking for — "sample 100 real failures and
hand-label each one against the fault-isolation tree" — except the labelling is automatic
because we have gold evidence.


In [ ]:
from collections import Counter

verdicts = Counter()
examples = {}
for r in rows:
    if r["is_null"] or r["answer_correct"] >= 1.0:
        continue
    x = next(y for y in bundle.questions if y.qid == r["qid"])
    c, _ = isolate(x, pipe, pipe.chunks)
    d = catalog.FAULT_ISOLATION.decide(c)
    owner = d["owner"] or "Label / question / rubric"
    verdicts[owner] += 1
    examples.setdefault(owner, x.query)

total = sum(verdicts.values())
viz.bars(list(verdicts.keys()), [v for v in verdicts.values()],
         title=f"Where {total} failing answers actually break",
         kicker="Fault isolation over the full eval set",
         ylabel="failing questions",
         caption="This distribution is the four-week plan. A room that argues about embedding "
                 "models before drawing this chart is arguing about the wrong stage.")

tables.show(pd.DataFrame(
    [[o, n, f"{n/total:.0%}", examples[o][:70]] for o, n in verdicts.most_common()],
    columns=["Owning stage", "Failures", "Share", "Example question"]),
    title="The work plan the tree produces",
    kicker="Failure distribution",
    caption="70% retrieval misses is a chunking and hybrid problem. 70% grounding failures is "
            "a prompt and abstention problem. Those are completely different four-week plans.",
    emphasize="Share")

A third of the failures land on the tree's **default** branch — "the pipeline is correct,
suspect the label, the question, or the rubric" — and the tree is right. Our
`answer_correct` is crude key-token matching, and it cannot recognise a correct answer that
has been quoted rather than phrased. The rubric is the broken component, not the retriever.

Do not skip past that. Ambiguous gold answers and weak scoring rubrics are the most
under-reported source of "regressions" in real engagements, and the reason a good tree ends
in *suspect your own measurement* rather than in *ship it*. Notebook 06 replaces this rubric
with a judge and re-attributes these cases.

---

## 1.7 The four failure signatures, reproduced

The deck lists four failure points for section 1. Learn them by their symptom, not their
cause — in production you see the symptom first. Each one below is reproduced on a real query
against our corpus, not described.


In [ ]:
sig = []

# 1. Lexical gap: the query and the answer passage share no terms.
gapq = next(x for x in bundle.questions if "pause between" in x.query)
gm, _ = metrics.resolve_gold(gapq, pipe.chunks)
lex = index.lexical(gapq.query, n=8)
den = retrieve.DenseRetriever(index, pipe.embedder).search(gapq.query, 8, cfg)
sig.append(["Lexical gap", gapq.query[:58],
            f"BM25 finds {metrics.evidence_recall_at_k([h.chunk_id for h in lex], gm):.2f} of "
            f"the gold, dense finds "
            f"{metrics.evidence_recall_at_k([h.chunk_id for h in den], gm):.2f}",
            "The question says 'pause / reconnection attempts'; the answer says 'retry delay'. "
            "No shared term for BM25 to match on."])

# 2. Missing hop: one of two required documents never enters top-k.
half = [r for r in rows if r["gold_items"] >= 2 and 0 < r["evidence_recall"] < 1.0]
hq = next(x for x in bundle.questions if x.qid == half[0]["qid"])
sig.append(["Missing hop", hq.query[:58],
            f"{half[0]['evidence_recall']:.2f} evidence recall, full-chain "
            f"{half[0]['full_chain_recall']:.0f}",
            f"{len(half)} of {len(answerable)} answerable questions land here: half the "
            "evidence chain arrives and the answer is built on it anyway."])

# 3. Distractor dominance: a high-scoring irrelevant passage takes a packing slot.
dq = next(x for x in bundle.questions if x.qid ==
          min((r for r in answerable if r["context_precision"] > 0),
              key=lambda r: r["context_precision"])["qid"])
dtr = pipe.run(dq.query, qid=dq.qid)
dgm, _ = metrics.resolve_gold(dq, pipe.chunks)
dgold = {c for s in dgm.values() for c in s}
sig.append(["Distractor dominance", dq.query[:58],
            f"{sum(1 for c in dtr.packed_ids if c not in dgold)}/{len(dtr.packed)} packed "
            "chunks carry no gold evidence",
            "Market-commentary articles name the same entities in the same language and "
            "decide nothing. They score well and cost gold chunks their slots."])

# 4. Metric blind spot: answer accuracy stable while evidence recall regresses.
narrow = pipeline.evaluate(pipe.variant("narrow", n_candidates=15), sample, pipe.chunks,
                           personas=bundle.personas)
wide = [r for r in rows if r["qid"] in {s["qid"] for s in narrow}]
sig.append(["Metric blind spot", "N=100 → N=15, same prompt, same k",
            f"answer correctness "
            f"{sum(r['answer_correct'] for r in wide)/len(wide):.3f} → "
            f"{sum(r['answer_correct'] for r in narrow)/len(narrow):.3f}, but Evidence "
            f"Recall@N {sum(r['evidence_recall_at_N'] for r in wide)/len(wide):.3f} → "
            f"{sum(r['evidence_recall_at_N'] for r in narrow)/len(narrow):.3f}",
            "Answer-level metrics move slowly and late. The retrieval metric moves first, "
            "which is the only reason you catch this before a customer does."])

tables.show(pd.DataFrame(sig, columns=["Signature", "Reproduced on", "Measured here",
                                       "Why it happens"]),
            title="The four section-1 failure signatures, reproduced on real queries",
            kicker="Failure points",
            caption="Every row is a measurement from this corpus, not a description. "
                    "Recognise them by symptom: that is what you get in production.",
            source="Deck slide 12", emphasize="Signature")

---

## 1.8 Case study: contextual retrieval

Anthropic published this in September 2024. The interesting part is not the technique — it is
*where* the fix lives. A chunk reading "revenue grew 3% over the previous quarter" is
unretrievable for the query "ACME Q2 2023 revenue growth", because the chunk never names the
company or the quarter. The reported fix runs at index time: prepend a short generated
description that situates each chunk inside its parent document, then embed and BM25-index
the augmented chunk.

Reported effect on failed retrievals: **−49%** with contextual embeddings plus contextual
BM25, and **−67%** with a reranking stage added.

Our `chunking.contextual` implements the same recipe. Offline the situating sentence comes
from a deterministic template; pass `describe=` a model — or a Bedrock generator — to produce
it the way you would ship it. Either way the cost lands at index time, once per chunk
version, rather than on every query forever.


In [ ]:
from raglab import chunking, embed, store

def build(strategy, **kw):
    chunks = chunking.chunk_corpus(bundle.documents, strategy=strategy, **kw)
    em = embed.LsaEmbedder(dim=96).fit([d.title + "\n" + d.body for d in bundle.documents])
    idx = store.InMemoryIndex()
    idx.upsert(chunks, em.encode_documents([c.text for c in chunks]), "v1", em.info.tag)
    p = pipeline.RagPipeline(idx, em, retrieve.RetrievalConfig(
        n_candidates=100, k=8, fusion="weighted", alpha=0.3, rerank="cross"), name=strategy)
    return chunks, p

runs = {}
for strat in ("structural", "contextual"):
    ch, pp = build(strat)
    runs[strat] = pipeline.evaluate(pp, sample, ch, personas=bundle.personas)

base = sum(1 - r["evidence_recall"] for r in runs["structural"]) / len(sample)
ctxr = sum(1 - r["evidence_recall"] for r in runs["contextual"]) / len(sample)
print(f"failed-retrieval rate   structural {base:.3f} → contextual {ctxr:.3f}   "
      f"({(ctxr - base) / base:+.1%})")
print(pipeline.compare_runs(runs, keys=("evidence_recall", "full_chain_recall",
                                        "context_precision")).to_string(index=False))

In [ ]:
tables.callout(
    "Our number is not Anthropic's, and it should not be. Their corpus, their encoder, their "
    "reranker, their queries. What transfers is the shape of the decision: <b>index-time "
    "compute is usually cheaper than query-time compute — you pay once per chunk, you pay per "
    "query forever</b>. Prompt caching is what made re-reading the whole document for every "
    "chunk affordable, and they kept BM25, because a widely-cited 'dense is enough' "
    "assumption did not survive contact with identifiers and exact terms."
    "<br><br><b>Ask in review:</b> what is our index-time budget per document, and when did "
    "we last re-run it?", kind="note", title="The engineering read")

---

## 1.9 The interview

These four are the section-1 baseline; the bank at the end of notebook 09 goes deeper. For
each one, the panel is listening for a *procedure*, not a definition.


In [ ]:
tables.show(pd.DataFrame([
    [catalog.SECTION_QUESTIONS[1][0],
     "Whether you know that 'retrieval failure' and 'generation failure' are four distinct "
     "verdicts, not two",
     "Walk the fault-isolation tree out loud. Name the trace fields each question needs. Say "
     "explicitly that you would not touch the prompt before Q1 answers yes."],
    [catalog.SECTION_QUESTIONS[1][1],
     "Whether you report the metric that predicts a correct multi-hop answer, or the one that "
     "flatters you",
     "Evidence Recall@k *and* full-chain recall. A system averaging 0.85 evidence recall can "
     "have full-chain recall of 0.55, and only the second number predicts correctness."],
    [catalog.SECTION_QUESTIONS[1][2],
     "Whether you understand that offline sets go stale and production has no labels",
     "Offline gates the release; production supplies the failures that become next quarter's "
     "regression cases. Name the feedback loop, not just the two lenses."],
    [catalog.SECTION_QUESTIONS[1][3],
     "Whether you treat the judge as a component that can regress",
     "Human-labelled calibration set, Cohen's κ rather than raw accuracy, judge–human "
     "agreement compared against human–human agreement, everything versioned. Notebook 06."],
], columns=["Question", "What the panel is testing", "What a strong answer covers"]),
    title="Typical interview questions: retrieval and evaluation",
    kicker="Section 1 · interview",
    caption="Notice that three of the four are answered by something you can now run rather "
            "than something you can recite.",
    source="Deck slide 14", emphasize="Question")

---

## 1.10 Checkpoint

Answer these against the system in front of you. The cell below checks your reasoning by
re-deriving each number — run it after you have committed to an answer.

1. If Evidence Recall@N is 0.78 at N=100, what is the highest end-to-end evidence recall any
   reranker could deliver?
2. You lower k from 8 to 3. Which of the four stages just changed, and which metric should
   move first?
3. Answer correctness is flat across a release but full-chain recall dropped 6 points. Which
   stage owns it, and what would you look at in the trace?


In [ ]:
k3 = pipeline.evaluate(pipe.variant("k=3", k=3), sample, pipe.chunks,
                       personas=bundle.personas)
k8 = [r for r in rows if r["qid"] in {x["qid"] for x in k3}]

def avg(rs, key):
    v = [r[key] for r in rs if r[key] is not None]
    return sum(v) / len(v)

print("1 ·  The ceiling is Recall@N itself. Reranking reorders the pool; it cannot add to it.")
print(f"     measured here: Recall@N = {avg(k8, 'evidence_recall_at_N'):.3f}  →  "
      f"no stage-2 model can take end-to-end evidence recall above that.\n")

print("2 ·  Stage three, packing. Nothing about retrieval changed — only how much survived.")
print(f"     Recall@N   k=8 {avg(k8, 'evidence_recall_at_N'):.3f}  →  k=3 "
      f"{avg(k3, 'evidence_recall_at_N'):.3f}   (unchanged, as it must be)")
print(f"     Recall@k   k=8 {avg(k8, 'evidence_recall'):.3f}  →  k=3 "
      f"{avg(k3, 'evidence_recall'):.3f}")
print(f"     full-chain k=8 {avg(k8, 'full_chain_recall'):.3f}  →  k=3 "
      f"{avg(k3, 'full_chain_recall'):.3f}   ← moves first and moves hardest\n")

print("3 ·  Ranking or packing (tree Q3). Full-chain recall falls when one hop of a two-hop")
print("     question stops surviving into the context, and answer-level metrics absorb that")
print("     slowly because single-hop questions still pass.")
print(f"     In the trace: compare candidate_ids against packed_ids for the affected "
      f"questions — 'retrieved then dropped' is the row you are looking for.")
print(f"     measured: dropping k from 8 to 3 costs "
      f"{avg(k8, 'full_chain_recall') - avg(k3, 'full_chain_recall'):.3f} full-chain recall "
      f"while answer correctness moves "
      f"{avg(k3, 'answer_correct') - avg(k8, 'answer_correct'):+.3f}.")

---

## What carries forward

- The four-stage model, and the habit of asking *which stage* before asking *what fix*.
- `Recall@N` is a ceiling. `Recall@k` is a delivery. The gap between them is precision work.
- Full-chain recall, not average evidence recall, is the number that predicts a correct
  multi-hop answer.
- The fault-isolation tree is a function you can run over a failure sample, and its output
  distribution is a work plan.

**Next:** `02_multihop_rag_use_case.ipynb` — what one dataset record lets you evaluate, why
question type drives retrieval strategy, and how to manufacture an eval set from a client
corpus that has no labels at all.
